In [ ]:
!pip install -q torch transformers sentence-transformers spacy pytextrank
!python -m spacy download en_core_web_sm

In [ ]:
import torch
import spacy
import pytextrank
from transformers import T5Tokenizer, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")
path = "Path_of_files_saved_in_your_laptop"
tokenizer = T5Tokenizer.from_pretrained(path)
model = T5ForConditionalGeneration.from_pretrained(path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
import numpy as np
def summary(text,maxi=8,lambda_param=0.7):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    if len(sentences) <= maxi:
        return " ".join(sentences)
    embeddings = embedder.encode(sentences)
    embeddings1 = embedder.encode([text])
    doc_sim = cosine_similarity(
        embeddings,
        embeddings1
    ).flatten()
    sentence = []
    idx = []
    idx1 = np.argmax(doc_sim)
    sentence.append(sentences[idx1])
    idx.append(idx1)
    while len(sentence) < maxi:
        scores = []
        for i in range(len(sentences)):
            if i in idx:
                scores.append(-1)
                continue
            new_sim = cosine_similarity(
                embeddings[i].reshape(1, -1),
                embeddings[idx]
            ).max()
            mmr = (
                lambda_param*doc_sim[i]-(1-lambda_param)*new_sim
            )
            scores.append(mmr)
        next_idx = np.argmax(scores)
        sentence.append(sentences[next_idx])
        idx.append(next_idx)
    return " ".join(sentence)

In [ ]:
def summarize(text):
    extracted = summary(text, maxi=8, lambda_param=0.7)
    inputs = tokenizer(
        "summarize: " + extracted,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)
    with torch.no_grad():
        output = model.generate(
            inputs["input_ids"],
            max_length=160,
            num_beams=4,
            no_repeat_ngram_size=3
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
text = """
TEXT_YOU_WANT_TO_SUMMARIZE
"""
ans = summarize(text)
print("SUMMARY:\n", ans)